# ⚖️ 08 — Building Custom Judges

Three ways to build a judge for the same job — `GEval`, `DAGMetric`, and a RAGAS rubric —
compared side by side on the same cases. Then the two bias tests that decide whether you can
believe any of them.

## Learning Objectives
In this notebook, you will learn:
1. **`GEval`** - criteria-driven scoring, and the `SingleTurnParams` API that replaced the now-deprecated `LLMTestCaseParams`
2. **`DAGMetric`** - a decision tree, for when 'rate this 1-5' is the wrong shape for the question
3. **RAGAS rubric metrics** - the same job again, and where the three disagree
4. **Reference-based versus reference-free** - and why only the second survives into production
5. **Bias tests** - position bias (swap the answers; does the verdict flip?) and same-model bias

## Prerequisites
- Notebooks 03, 04 and 07 (`data/golden_set.json`, `data/trajectories.json`)

---
## 🗺️ The concept

```
  ONE QUESTION: "is this support reply good?"

  GEval            criteria in prose -> the judge derives steps
                   flexible, fast to write, hardest to debug

  DAGMetric        an explicit decision tree
                   did it cite? -> yes -> is the citation right? -> yes -> ...
                   auditable, verbose, no room for the judge to improvise

  RubricsScore     fixed level descriptions
                   stable across runs, blunt at the boundaries

  BIAS TESTS (run these before trusting any of the three)
    position bias   score (A,B) then (B,A). Verdict should not flip.
    same-model bias gpt-4o-mini judging gpt-4o-mini rates itself high.
```

### Key Concepts:
- **A judge is a model, so it needs its own evaluation.** "We checked a few by hand" is not a
  validation strategy. Notebook 09 gives it human labels and an agreement number.
- **The three shapes fail differently.** Prose criteria drift between runs; a DAG cannot
  express nuance it was not given; a rubric is stable but blunt at the boundaries.
- **Reference-free judging is the only kind that survives deployment**, because production has
  no answer key. Notebook 12 depends on this property.

---
## 🔨 Build it

### 0.1 ⚙️ Setup

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: golden set, trajectories, the judged-score cache
# ============================================================================
import os
import json
import hashlib
import concurrent.futures
import asyncio
import threading
import warnings
import statistics
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
# DeepEval reports anonymous usage on the first metric it builds. That call is
# not on the critical path of an evaluation, but it happens inside it — so a
# slow one becomes a slow metric. Off, here and in CI.
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# DeepEval also wraps every model call in a timeout of its own, implemented as
# one worker thread per call. That wrapper is what stalls in this course's
# environment — the request underneath it is fast — so it is switched off. The
# OpenAI client keeps its own request timeout, and judge_bounded() below retries
# a call that really does fail.
os.environ.setdefault("DEEPEVAL_DISABLE_TIMEOUTS", "1")

load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
SCORE_CACHE = os.path.join(DATA_DIR, "score_cache")
os.makedirs(SCORE_CACHE, exist_ok=True)

with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

with open(os.path.join(DATA_DIR, "trajectories.json"), encoding="utf-8") as f:
    TRAJECTORIES = json.load(f)

CASES = GOLDEN["cases"]
CASE_BY_ID = {c["case_id"]: c for c in CASES}

SCORE_MODE = "auto"        # "auto"   = replay if cached, judge + cache on a miss
                           # "strict" = raise on a miss (use in CI, never spends)
                           # "record" = re-judge everything, ignoring the cache


def score_key(metric_name, payload):
    blob = json.dumps({"metric": metric_name, "payload": payload},
                      sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def _cache_path(metric_name, payload):
    return os.path.join(SCORE_CACHE, score_key(metric_name, payload) + ".json")


def _read_cache(path):
    if SCORE_MODE != "record" and os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return json.load(f)["score"]
    return None


def _miss(metric_name, path):
    """What to do when nothing is cached. `strict` refuses to spend money."""
    if SCORE_MODE == "strict":
        raise FileNotFoundError(
            "No cached score for " + metric_name + " (" + os.path.basename(path) + ").\n"
            "SCORE_MODE is 'strict', so nothing will be judged. Run the notebook "
            "with SCORE_MODE = 'auto' to fill the cache.")


def _write_cache(path, metric_name, payload, score):
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"metric": metric_name, "payload": payload, "score": score},
                  f, ensure_ascii=False, indent=2)
    return score


# The model that JUDGES, which is not the model the agent RUNS on. A judge is
# picked for reliability first and price second: gpt-4o-mini occasionally
# degenerates into a repetition loop under a strict JSON schema and generates
# until its token cap, which arrives as a 90-second stall rather than an error.
JUDGE_MODEL = "gpt-4.1-mini"

# Judge one case at a time. DeepEval's async path runs its own event loop per
# call; over a long sequential run that path can stall with the request in
# flight and no timeout firing. Nothing here wants concurrency anyway — the
# loops below are sequential so the printed table stays in case order.
JUDGE_ASYNC = False

JUDGE_TIMEOUT_SECONDS = 60   # a judge call slower than this is stuck, not slow
JUDGE_ATTEMPTS = 3           # ...and a stuck call is worth asking again


def judge_bounded(compute):
    """Call the judge under a hard ceiling, and retry a call that misses it.

    Judges are LLM calls and fail the way LLM calls fail; the worst of those
    failures is the one that never returns. Bounding it is the harness's job —
    "the suite never finished" is the least useful result CI can give you.
    A worker thread is safe here because every metric is built with
    async_mode=False, so nothing touches the notebook kernel's event loop.
    """
    for attempt in range(JUDGE_ATTEMPTS):
        pool = concurrent.futures.ThreadPoolExecutor(max_workers=1)
        try:
            return pool.submit(compute).result(timeout=JUDGE_TIMEOUT_SECONDS)
        except Exception as error:
            print("⚠️  judge call failed (%s) — attempt %d of %d"
                  % (type(error).__name__, attempt + 1, JUDGE_ATTEMPTS))
        finally:
            pool.shutdown(wait=False)   # a stuck worker is a daemon; let it go
    raise RuntimeError(
        "A judge call failed %d times in a row. Nothing was cached, so no case "
        "silently became a 0.0 — re-run the cell once the judge is healthy."
        % JUDGE_ATTEMPTS)


def run_async(coro):
    """Run a coroutine from ordinary synchronous code, notebook or not.

    RAGAS metrics are async-only, and a Jupyter kernel is already running an
    event loop of its own — which is why asyncio.run() alone raises "cannot be
    called from a running event loop" here. Running the coroutine on its own
    loop, in its own thread, works in both places.
    """
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)      # no loop in this thread: the plain path

    box = {}

    def target():
        box["value"] = asyncio.run(coro)

    thread = threading.Thread(target=target)
    thread.start()
    thread.join()
    return box["value"]

def cached_measure(metric_name, payload, compute):
    """Judge once, reuse forever. A cached verdict is what lets notebooks 10 and
    11 compare runs — a score that moves while nothing moved is not a measurement.
    """
    path = _cache_path(metric_name, payload)
    hit = _read_cache(path)
    if hit is not None:
        return hit
    _miss(metric_name, path)
    return _write_cache(path, metric_name, payload, judge_bounded(compute))

print("✅ Setup complete")
print("🗂️  %d cases, %d trajectories | SCORE_MODE = %s"
      % (len(CASES), len(TRAJECTORIES), SCORE_MODE))

✅ Setup complete
🗂️  40 cases, 25 trajectories | SCORE_MODE = auto


In [2]:
# ============================================================================
# THE REPLIES UNDER JUDGEMENT
# ============================================================================
# One judgeable item per run: the question, the reply, and the reference the
# golden set can offer. `reference` is deliberately kept separate — half of this
# notebook is about what happens when you do not have it.

JUDGEABLE = []
for traj in TRAJECTORIES:
    case = CASE_BY_ID[traj["case_id"]]
    answer = traj.get("final_answer") or ""
    if not answer:
        continue
    JUDGEABLE.append({
        "case_id": traj["case_id"],
        "question": case["input"],
        "answer": answer,
        "reference": " ".join(case["must_contain"]) or None,
        "expected_behaviour": case["expected_behaviour"],
        "strata": case["strata"],
    })

print("⚖️  judgeable replies: %d" % len(JUDGEABLE))
print("📎 of those, %d have a reference in the golden set"
      % sum(1 for j in JUDGEABLE if j["reference"]))

⚖️  judgeable replies: 25
📎 of those, 11 have a reference in the golden set


---
## 1️⃣ Judge one: `GEval`

`GEval` takes criteria in prose and lets the judge derive its own evaluation steps. It is the
fastest of the three to write and the hardest to debug — when a score looks wrong, there is no
branch to point at, only a paragraph to re-word.

> **API notes (DeepEval 4.2.x)**
> - `evaluation_params` takes **`SingleTurnParams`**, which replaced `LLMTestCaseParams`. Older
>   code that imports `LLMTestCaseParams` still runs on 4.2.3, with a deprecation warning: it is
>   an alias that a future release will remove.
> - Give it **either** `criteria` (prose, judge derives steps) **or** `evaluation_steps`
>   (you write the steps). `evaluation_steps` is more reproducible; `criteria` is faster.
> - An optional `rubric=[Rubric(score_range=..., expected_outcome=...)]` pins what each band
>   means, which reduces drift between runs.

In [3]:
# ============================================================================
# GEVAL: prose criteria, explicit steps, and a rubric to pin the bands
# ============================================================================
# Written as evaluation_steps rather than free-form criteria, because steps are
# what you will actually argue about in review.

def build_geval():
    from deepeval.metrics import GEval
    from deepeval.metrics.g_eval.utils import Rubric
    from deepeval.test_case import SingleTurnParams

    return GEval(
        name="support_reply_quality",
        evaluation_steps=[
            "Check whether the reply answers the customer's actual question, not a "
            "nearby one.",
            "Check whether every factual claim is attributable to the retrieved "
            "documentation. An uncited claim about policy is a failure.",
            "Check whether the reply states clearly what will happen next, if "
            "anything is pending (an approval, a refund, a callback).",
            "A refusal is CORRECT when the documentation does not cover the "
            "question. Do not penalise it.",
        ],
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
        rubric=[
            Rubric(score_range=(0, 3),
                   expected_outcome="Wrong, or confidently unsupported."),
            Rubric(score_range=(4, 7),
                   expected_outcome="Broadly right but vague, or missing next steps."),
            Rubric(score_range=(8, 10),
                   expected_outcome="Answers the question, grounded, and says what "
                                    "happens next."),
        ],
        model=JUDGE_MODEL,
        async_mode=JUDGE_ASYNC,
        threshold=0.7,
    )


def geval_score(item):
    payload = {"question": item["question"], "answer": item["answer"]}

    def compute():
        from deepeval.test_case import LLMTestCase
        metric = build_geval()
        metric.measure(LLMTestCase(input=item["question"],
                                   actual_output=item["answer"]))
        return float(metric.score)

    return cached_measure("geval_reply_quality", payload, compute)


GEVAL = [geval_score(j) for j in JUDGEABLE]
print("1️⃣  GEval mean: %.3f" % statistics.mean(GEVAL))

1️⃣  GEval mean: 0.702


---
## 2️⃣ Judge two: `DAGMetric`

The same question as a decision tree. Every path ends at a fixed score, so two runs that take
the same branches *must* produce the same number — the judge has no room to improvise.

That is the trade. A DAG cannot express a nuance you did not anticipate, and adding one means
adding a branch. In exchange you get an answer to *"why did this score 4?"* that is a path
rather than an opinion.

In [4]:
# ============================================================================
# DAG: the same judgement, as branches instead of prose
# ============================================================================
#   does the reply answer the question asked?
#     |no -> 0
#     |yes -> is every policy claim cited?
#               |no -> 3
#               |yes -> does it say what happens next?
#                         |no -> 7
#                         |yes -> 10

def build_dag_judge():
    from deepeval.metrics.dag import (DeepAcyclicGraph, BinaryJudgementNode,
                                      VerdictNode)
    from deepeval.metrics import DAGMetric
    from deepeval.test_case import SingleTurnParams

    says_next = BinaryJudgementNode(
        criteria=("Does the reply state what happens next where something is "
                  "pending — an approval, a refund being processed, a callback? "
                  "If nothing is pending, answer yes."),
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        children=[VerdictNode(verdict=True, score=10),
                  VerdictNode(verdict=False, score=7)],
    )

    cited = BinaryJudgementNode(
        criteria=("Is every claim about company policy attributed to a document, "
                  "for example with a bracketed article id? A reply that makes no "
                  "policy claims at all counts as yes."),
        evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
        children=[VerdictNode(verdict=True, child=says_next),
                  VerdictNode(verdict=False, score=3)],
    )

    answers = BinaryJudgementNode(
        criteria=("Does the reply address the question the customer actually "
                  "asked? An explicit, correct refusal counts as addressing it."),
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
        children=[VerdictNode(verdict=True, child=cited),
                  VerdictNode(verdict=False, score=0)],
    )

    return DAGMetric(name="support_reply_dag",
                     dag=DeepAcyclicGraph(root_nodes=[answers]),
                     model=JUDGE_MODEL,
                     async_mode=JUDGE_ASYNC,
                     threshold=0.7)


def dag_score(item):
    payload = {"question": item["question"], "answer": item["answer"]}

    def compute():
        from deepeval.test_case import LLMTestCase
        metric = build_dag_judge()
        metric.measure(LLMTestCase(input=item["question"],
                                   actual_output=item["answer"]))
        return float(metric.score)

    return cached_measure("dag_reply_quality", payload, compute)


DAG = [dag_score(j) for j in JUDGEABLE]
print("2️⃣  DAG mean: %.3f" % statistics.mean(DAG))

2️⃣  DAG mean: 0.776


---
## 3️⃣ Judge three: RAGAS rubrics — and the reference question

RAGAS splits rubric scoring along exactly the axis this notebook cares about:

| Class | Needs an answer key? | Where it can run |
|---|---|---|
| `RubricsScoreWithReference` | **Yes** | Offline only — your golden set has the reference |
| `RubricsScoreWithoutReference` | No | Offline **and in production** |

> **API note.** There is no plain `RubricsScore` in `ragas.metrics.collections`. The legacy
> `from ragas.metrics import RubricsScore` still imports but is deprecated, and its own
> deprecation message points at `collections` — where the name does not exist. Use the two
> explicit classes.

This split is the whole lesson. Everything you can only measure *with* a reference stops
working the moment you deploy.

In [5]:
# ============================================================================
# RAGAS RUBRICS: the same job, with and without an answer key
# ============================================================================
# TODO (yours): these level descriptions are the judge. Write them.
#
# Worth deciding before you type:
#   - Does a correct refusal score at the TOP of the scale, or the middle? (If
#     the middle, your suite quietly teaches the agent to guess.)
#   - Is a reply that is right but uncited a 3 or a 1? That single choice decides
#     whether citation is a nice-to-have or a requirement.
#   - Should the WITHOUT-reference rubric be identical to the WITH-reference one?
#     It cannot check factual agreement, so scoring them the same way makes the
#     two numbers incomparable — which defeats the comparison below.

RUBRICS = {
    "score1_description": (
        "TODO: describe a clearly BAD reply — e.g. it answers a different question, "
        "or makes a confident policy claim with nothing behind it."
    ),
    "score2_description": (
        "TODO: describe a WEAK reply — e.g. broadly correct but vague, or correct "
        "and uncited."
    ),
    "score3_description": (
        "TODO: describe a GOOD reply — e.g. answers what was asked, grounded in "
        "the documentation, and explicit about anything still pending."
    ),
}


def build_rubric_metrics():
    """Built on a cache miss — both need a judge."""
    from openai import AsyncOpenAI
    from ragas.llms.base import llm_factory
    from ragas.metrics.collections import (RubricsScoreWithReference,
                                           RubricsScoreWithoutReference)
    judge = llm_factory("gpt-4o-mini", client=AsyncOpenAI())
    return (RubricsScoreWithReference(llm=judge, rubrics=RUBRICS),
            RubricsScoreWithoutReference(llm=judge, rubrics=RUBRICS))


def rubric_scores(item):
    """Return (with_reference_or_None, without_reference)."""
    base = {"question": item["question"], "answer": item["answer"]}

    def compute_without():
        _, without_ref = build_rubric_metrics()
        return float(run_async(without_ref.ascore(
            user_input=item["question"], response=item["answer"])).value)

    without = cached_measure("rubric_without_reference", base, compute_without)

    if not item["reference"]:
        return None, without

    def compute_with():
        with_ref, _ = build_rubric_metrics()
        return float(run_async(with_ref.ascore(
            user_input=item["question"], response=item["answer"],
            reference=item["reference"])).value)

    with_ref_score = cached_measure(
        "rubric_with_reference", dict(base, reference=item["reference"]),
        compute_with)
    return with_ref_score, without


RUBRIC = [rubric_scores(j) for j in JUDGEABLE]
with_ref = [r[0] for r in RUBRIC if r[0] is not None]
without_ref = [r[1] for r in RUBRIC]

print("3️⃣  rubric WITH    reference: %.3f (on %d cases)"
      % (statistics.mean(with_ref), len(with_ref)) if with_ref else "   (no references)")
print("3️⃣  rubric WITHOUT reference: %.3f (on %d cases)"
      % (statistics.mean(without_ref), len(without_ref)))

3️⃣  rubric WITH    reference: 3.455 (on 11 cases)
3️⃣  rubric WITHOUT reference: 3.600 (on 25 cases)


---
### 3.1 🎯 Few-shot calibration — show the judge where the boundaries are

A rubric says what each level means. It does not show the judge where the boundaries fall,
and the boundaries are exactly where two runs disagree. **Few-shot calibration examples** do
show it: a handful of solved cases inside the judge prompt, each with its verdict and one line
of why. Judges tend to follow the pattern of their examples more closely than prose.

Choose examples that sit **on the boundary**, not in the easy middle. For a groundedness judge
the boundary is *strict deducibility* — can the reply be derived from the documentation alone,
whether or not it is true:

| Case | Verdict | The point it pins down |
|---|---|---|
| Answers half the question, all of it from the docs | 1 | completeness is a different metric |
| Wrong in the world, but matches the docs | 1 | the judge scores against the docs, not the truth |
| Right in the world, absent from the docs | 0 | a correct memory is still not grounded |
| The docs hold a bare fragment (`30.`) | 0 | matching a token is not deducing an answer |
| The docs are about something else | 0 | the obvious case, kept as an anchor |

> **Note**: the examples become part of every judgement, so they carry their own bias. Five
> English examples can make a judge stricter in Spanish and Hindi — the by-stratum
> table at the end of this notebook is where that would show. And keep them out of the golden
> set, or the judge has seen the answers.

The cell writes the prompt only. Nothing in it calls a model.

In [ ]:
# ============================================================================
# FEW-SHOT JUDGE PROMPT: calibration examples on the strict-deducibility edge
# ============================================================================
# Fill the three slots with .format(question=..., context=..., answer=...) and
# send it to any judge model that returns a 0 or a 1.

GROUNDED_FEW_SHOT_PROMPT = """You judge one thing: can the REPLY be deduced from the DOCUMENTATION alone?
Whether the reply is true in the real world does not matter, and background knowledge does
not count. Answer 1 if every claim in the reply follows from the documentation, otherwise 0.

Example 1 -> 1
Question: How long do I have to dispute a charge, and how long do refunds take?
Documentation: Charges can be disputed within 30 days of the statement date.
Reply: You have 30 days to dispute a charge.
Why: it answers only half the question, but everything it says is in the documentation.

Example 2 -> 1
Question: How long is a password-reset link valid?
Documentation: Password-reset links expire after 24 hours.
Reply: The reset link is valid for 24 hours.
Why: the live product may say otherwise, but the reply matches the documentation it was given.

Example 3 -> 0
Question: How long is a password-reset link valid?
Documentation: You can request a password-reset link from the sign-in page.
Reply: The reset link is valid for 60 minutes.
Why: the number may be right, but it came from memory; the documentation never states it.

Example 4 -> 0
Question: How long do I have to dispute a charge?
Documentation: 30.
Reply: You have 30 days.
Why: the documentation never says what the 30 refers to, so the reply cannot be deduced from it.

Example 5 -> 0
Question: Can I change my data region after signup?
Documentation: Two-factor authentication supports TOTP authenticator apps.
Reply: No, the data region is fixed once your account is created.
Why: nothing in the documentation is about data regions.

Now judge this one.
Question: {question}
Documentation: {context}
Reply: {answer}
Score (0 or 1):"""

print("🎯 few-shot judge prompt: %d calibration examples, %d characters"
      % (GROUNDED_FEW_SHOT_PROMPT.count("\nExample "), len(GROUNDED_FEW_SHOT_PROMPT)))

---
## 🔍 Part 4: Where the three disagree

Three judges, one question, the same replies. Agreement is not the goal — **understanding the
disagreement is**. A case all three score highly tells you nothing. A case they split on is
either an ambiguous reply or an ambiguous rubric, and both are worth knowing about.

In [6]:
# ============================================================================
# COMPARE: normalise to 0-1 and put the three side by side
# ============================================================================
# GEval and DAG already return 0-1. RAGAS rubrics return the rubric's own scale
# (1-3 here), so they need dividing before any comparison is meaningful. This is
# the most common quiet bug in judge comparisons.

RUBRIC_MAX = 3.0

COMPARISON = []
for i, item in enumerate(JUDGEABLE):
    with_ref_raw, without_ref_raw = RUBRIC[i]
    COMPARISON.append({
        "case_id": item["case_id"],
        "strata": item["strata"],
        "expected_behaviour": item["expected_behaviour"],
        "geval": GEVAL[i],
        "dag": DAG[i],
        "rubric_with_ref": (None if with_ref_raw is None
                            else with_ref_raw / RUBRIC_MAX),
        "rubric_without_ref": without_ref_raw / RUBRIC_MAX,
    })


def spread(row):
    values = [row["geval"], row["dag"], row["rubric_without_ref"]]
    return max(values) - min(values)


COMPARISON.sort(key=spread, reverse=True)

print("%-10s %8s %8s %10s %10s %8s" % (
    "case", "geval", "dag", "rub(ref)", "rub(free)", "spread"))
for r in COMPARISON[:12]:
    print("%-10s %8.2f %8.2f %10s %10.2f %8.2f" % (
        r["case_id"], r["geval"], r["dag"],
        "-" if r["rubric_with_ref"] is None else "%.2f" % r["rubric_with_ref"],
        r["rubric_without_ref"], spread(r)))
print()
wide = [r for r in COMPARISON if spread(r) > 0.3]
print("🔀 %d of %d cases where the judges differ by more than 0.30"
      % (len(wide), len(COMPARISON)))
print("   These are your review queue. Notebook 09 sends exactly these to a human.")

case          geval      dag   rub(ref)  rub(free)   spread
GC-021         0.22     1.00       1.33       1.67     1.44
GC-017         0.38     0.30       1.67       1.67     1.37
GC-037         0.87     0.30       1.33       1.67     1.37
GC-039         0.90     0.30          -       1.67     1.37
GC-024         0.69     1.00       1.00       1.67     0.98
GC-029         0.40     0.70       1.67       1.33     0.93
GC-026         0.84     1.00       1.00       1.67     0.82
GC-023         0.85     1.00          -       1.67     0.82
GC-015         0.89     1.00       1.00       1.67     0.77
GC-031         0.90     1.00          -       1.67     0.77
GC-040         0.60     0.30          -       1.00     0.70
GC-022         0.66     1.00          -       1.33     0.68

🔀 21 of 25 cases where the judges differ by more than 0.30
   These are your review queue. Notebook 09 sends exactly these to a human.


In [7]:
# ============================================================================
# THE REFERENCE GAP: what you lose when the answer key goes away
# ============================================================================
# Every case here has both numbers, so the difference is attributable to the
# reference and nothing else.

paired = [r for r in COMPARISON if r["rubric_with_ref"] is not None]
if paired:
    gaps = [r["rubric_with_ref"] - r["rubric_without_ref"] for r in paired]
    print("📎 cases scored both ways: %d" % len(paired))
    print("📎 mean WITH reference   : %.3f"
          % statistics.mean(r["rubric_with_ref"] for r in paired))
    print("📎 mean WITHOUT reference: %.3f"
          % statistics.mean(r["rubric_without_ref"] for r in paired))
    print("📎 mean signed gap       : %+.3f" % statistics.mean(gaps))
    print("📎 mean absolute gap     : %.3f" % statistics.mean(abs(g) for g in gaps))
    print()
    print("📌 The absolute gap is the number that matters. It is how much of your")
    print("   offline signal simply does not exist in production, where nobody has")
    print("   written down the right answer. Notebook 12 monitors only what is left.")

    by_behaviour = defaultdict(list)
    for r in paired:
        by_behaviour[r["expected_behaviour"]].append(
            abs(r["rubric_with_ref"] - r["rubric_without_ref"]))
    print()
    print("%-16s %5s %10s" % ("behaviour", "n", "abs gap"))
    for key in sorted(by_behaviour):
        values = by_behaviour[key]
        print("%-16s %5d %10.3f" % (key, len(values), statistics.mean(values)))
else:
    print("(no cases carry a reference — add `must_contain` in notebook 03)")

📎 cases scored both ways: 11
📎 mean WITH reference   : 1.152
📎 mean WITHOUT reference: 1.303
📎 mean signed gap       : -0.152
📎 mean absolute gap     : 0.333

📌 The absolute gap is the number that matters. It is how much of your
   offline signal simply does not exist in production, where nobody has
   written down the right answer. Notebook 12 monitors only what is left.

behaviour            n    abs gap
answer              11      0.333


---
## 🎭 Part 5: Bias test one — position bias

Ask a judge to pick between answer A and answer B. Then ask again with the order swapped.

**The verdict should not change.** When it does, the judge is scoring presentation order rather
than quality, and every pairwise number it has ever produced is suspect — including the canary
comparisons notebook 12 uses to decide on a rollback.

> **API note.** DeepEval's pairwise judge is `ArenaGEval` over an
> `ArenaTestCase(contestants=[Contestant(name=..., test_case=...), ...])`. It returns a winner
> by name, not a score.

In [8]:
# ============================================================================
# POSITION BIAS: the same pair, judged twice, in both orders
# ============================================================================

def build_arena():
    from deepeval.metrics import ArenaGEval
    from deepeval.test_case import SingleTurnParams
    return ArenaGEval(
        name="better_support_reply",
        criteria=("Choose the reply that better answers the customer's question "
                  "while staying grounded in the documentation. Prefer a correct "
                  "refusal over a confident guess."),
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
        model=JUDGE_MODEL,
        async_mode=JUDGE_ASYNC,
    )


def judge_pair(question, first_name, first_answer, second_name, second_answer):
    """Return the winning contestant's name, cached by the exact ordering."""
    payload = {"question": question,
               "order": [first_name, second_name],
               "answers": [first_answer, second_answer]}

    def compute():
        from deepeval.test_case import (ArenaTestCase, Contestant, LLMTestCase)
        metric = build_arena()
        metric.measure(ArenaTestCase(contestants=[
            Contestant(name=first_name,
                       test_case=LLMTestCase(input=question,
                                             actual_output=first_answer)),
            Contestant(name=second_name,
                       test_case=LLMTestCase(input=question,
                                             actual_output=second_answer)),
        ]))
        return metric.winner

    return cached_measure("arena_winner", payload, compute)


def position_bias(question, answer_a, answer_b):
    """Judge both orders. Returns (winner_ab, winner_ba, flipped)."""
    forward = judge_pair(question, "A", answer_a, "B", answer_b)
    reverse = judge_pair(question, "B", answer_b, "A", answer_a)
    return forward, reverse, forward != reverse


print("✅ position_bias() ready — two judge calls per pair, both cached")

✅ position_bias() ready — two judge calls per pair, both cached


In [9]:
# ============================================================================
# RUN THE POSITION-BIAS PROBE on pairs the judges already disagreed about
# ============================================================================
# Pair each high-spread reply against a deliberately weaker variant of itself:
# same content, hedged and uncited. A judge with no position bias should pick
# the original every time, in both orders.

def weaken(answer):
    """A worse reply with the same information: hedged, uncited, no next step."""
    stripped = answer
    for marker in ["[", "]"]:
        stripped = stripped.replace(marker, "")
    return "I think " + stripped.split(".")[0].strip().lower() + ", but I'm not certain."


PAIRS = [(r["case_id"], j) for r, j in
         ((r, next(x for x in JUDGEABLE if x["case_id"] == r["case_id"]))
          for r in COMPARISON[:6])]

flips = 0
print("%-10s %10s %10s %8s" % ("case", "A,B ->", "B,A ->", "flipped"))
for case_id, item in PAIRS:
    strong = item["answer"]
    weak = weaken(strong)
    forward, reverse, flipped = position_bias(item["question"], strong, weak)
    flips += 1 if flipped else 0
    print("%-10s %10s %10s %8s" % (case_id, forward, reverse,
                                   "⚠️  YES" if flipped else "no"))

print()
print("🎭 verdict flipped on %d of %d pairs" % (flips, len(PAIRS)))
if flips:
    print("   ⚠️  This judge is measuring order. Fix it before trusting any")
    print("       pairwise number: average both orders, or score each answer")
    print("       independently instead of comparing them.")
else:
    print("   ✅ stable under swapping — necessary, not sufficient.")

case           A,B ->     B,A ->  flipped
GC-021              A          A       no
GC-017              A          A       no
GC-037              A          A       no
GC-039              A          A       no
GC-024              A          A       no
GC-029              A          A       no

🎭 verdict flipped on 0 of 6 pairs
   ✅ stable under swapping — necessary, not sufficient.


---
### 5.1 📏 Bias test one-and-a-half — was it the quality, or just the length?

Part 5 cleared this judge of position bias: the strong reply won in both orders, every time.
That is necessary and **not sufficient**, because it leaves a second explanation standing.

The weakened replies are not only worse — they are a different *length*. So "the strong one
won" and "the longer one won" are the same sentence about this data, and the probe above
cannot tell them apart.

**Verbosity bias is the judge preferring the longer answer.** The guide that this course
audits against names it alongside position bias, and it is the one that quietly inflates a
canary comparison: ship a chattier build and the win rate rises without the answers improving.

Two cells. The first costs nothing — it re-reads verdicts already cached and reports the
length distribution next to the win rate, which is the reporting rule this course argues for.
The second is the actual controlled test and needs fresh judge calls, so it is behind a
switch.

In [10]:
# ============================================================================
# LENGTH AUDIT: report the length distribution next to the win rate
# ============================================================================
# Free. position_bias() replays the verdicts Part 5 already cached, so this
# spends nothing and returns the same answers every run.

def words(text):
    return len(text.split())


winner_lengths = []
loser_lengths = []
strong_wins = 0
longer_wins = 0
comparisons = 0

for case_id, item in PAIRS:
    strong = item["answer"]
    weak = weaken(strong)
    forward, reverse, _ = position_bias(item["question"], strong, weak)

    # Both calls label the original "A" and the weakened variant "B"; only the
    # order they are shown in changes. So winner == "A" means the strong reply won.
    for winner in (forward, reverse):
        comparisons += 1
        won, lost = (strong, weak) if winner == "A" else (weak, strong)
        winner_lengths.append(words(won))
        loser_lengths.append(words(lost))
        strong_wins += 1 if winner == "A" else 0
        longer_wins += 1 if words(won) > words(lost) else 0

mean_win = statistics.mean(winner_lengths)
mean_lose = statistics.mean(loser_lengths)

print("📏 length audit over %d cached comparisons" % comparisons)
print("   strong reply won      : %d of %d" % (strong_wins, comparisons))
print("   mean words, winners   : %.1f" % mean_win)
print("   mean words, losers    : %.1f" % mean_lose)
print("   longer answer won     : %d of %d" % (longer_wins, comparisons))
print()
print("   winners are %.0f%% longer on average." % (100.0 * (mean_win - mean_lose) / mean_lose))
print("   ⚠️  Quality and length move together here, so this probe cannot say")
print("       which one the judge responded to. That is what 5.1's second cell")
print("       exists to separate — and why a win rate published without a length")
print("       distribution beside it is partly a word count.")

📏 length audit over 12 cached comparisons
   strong reply won      : 12 of 12
   mean words, winners   : 25.2
   mean words, losers    : 18.7
   longer answer won     : 6 of 12

   winners are 35% longer on average.
   ⚠️  Quality and length move together here, so this probe cannot say
       which one the judge responded to. That is what 5.1's second cell
       exists to separate — and why a win rate published without a length
       distribution beside it is partly a word count.


In [11]:
# ============================================================================
# LENGTH-CONTROLLED PROBE: make the WORSE answer the LONGER one
# ============================================================================
# This asks the judge something it has not been asked, so it cannot be replayed.
# Off by default: flip the switch to spend roughly 12 fresh judge calls.

RUN_LENGTH_PROBE = False


def pad_to_length(text, target_words):
    """Same information, more words. Filler a judge should not reward."""
    filler = [
        "Please let me know if you would like me to look into this further.",
        "I am happy to help with anything else you need today.",
        "Thank you for your patience while I checked this for you.",
    ]
    out = text
    for sentence in filler:
        if len(out.split()) >= target_words:
            break
        out = out + " " + sentence
    return out


if not RUN_LENGTH_PROBE:
    print("⏭️  skipped — set RUN_LENGTH_PROBE = True to run it (~12 judge calls)")
    print()
    print("   What it does: pad the WEAKENED reply with content-free filler until")
    print("   it is the longer of the two, then judge again. The strong reply is")
    print("   still the better answer, so it should still win.")
    print()
    print("   Reading the result:")
    print("     strong still wins  -> length was not doing the work in Part 5")
    print("     padded reply wins  -> the judge is scoring words, and every")
    print("                           pairwise number in this course is suspect,")
    print("                           including notebook 12's canary win rate")
else:
    flipped_by_length = 0
    print("%-10s %8s %8s %10s" % ("case", "strong", "padded", "winner"))
    for case_id, item in PAIRS:
        strong = item["answer"]
        padded = pad_to_length(weaken(strong), words(strong) + 10)
        forward, reverse, _ = position_bias(item["question"], strong, padded)
        for winner in (forward, reverse):
            flipped_by_length += 1 if winner == "B" else 0
        print("%-10s %8d %8d %10s" % (case_id, words(strong), words(padded),
                                      forward + "/" + reverse))

    print()
    print("📏 the longer-but-worse reply won %d of %d comparisons"
          % (flipped_by_length, 2 * len(PAIRS)))
    if flipped_by_length:
        print("   ⚠️  Verbosity bias confirmed. Control for length before")
        print("       reporting any win rate: match the arms, or report the")
        print("       length distribution beside the number.")
    else:
        print("   ✅ Length did not buy a win. Part 5's result stands on quality.")

⏭️  skipped — set RUN_LENGTH_PROBE = True to run it (~12 judge calls)

   What it does: pad the WEAKENED reply with content-free filler until
   it is the longer of the two, then judge again. The strong reply is
   still the better answer, so it should still win.

   Reading the result:
     strong still wins  -> length was not doing the work in Part 5
     padded reply wins  -> the judge is scoring words, and every
                           pairwise number in this course is suspect,
                           including notebook 12's canary win rate


---
## 🪞 Part 6: Bias test two — same-model bias

A judge from the same family as the generator rates that generator's output generously. It is
recognising its own habits — phrasing, structure, hedging — and reading familiarity as quality.

The test is to score the same replies with a judge from a **different** family and compare. You
are not looking for identical numbers. You are looking for whether the *ranking* survives,
because ranking is what a release gate acts on.

In [12]:
# ============================================================================
# SAME-MODEL BIAS: re-score with a different family, compare the RANKING
# ============================================================================
# Swap JUDGE_B for any model you have access to. The point is a different
# family, not a bigger model — gpt-4o judging gpt-4o-mini is still one family.

JUDGE_A = JUDGE_MODEL          # the same family as the other judges here
JUDGE_B = JUDGE_MODEL          # <- change me, e.g. a Claude or Llama model


def geval_score_with(item, model_name):
    payload = {"question": item["question"], "answer": item["answer"],
               "judge": model_name}

    def compute():
        from deepeval.metrics import GEval
        from deepeval.test_case import LLMTestCase, SingleTurnParams
        metric = GEval(
            name="support_reply_quality",
            evaluation_steps=build_geval().evaluation_steps,
            evaluation_params=[SingleTurnParams.INPUT,
                               SingleTurnParams.ACTUAL_OUTPUT],
            model=model_name,
            async_mode=JUDGE_ASYNC,
        )
        metric.measure(LLMTestCase(input=item["question"],
                                   actual_output=item["answer"]))
        return float(metric.score)

    return cached_measure("geval_by_judge", payload, compute)


def rank_correlation(xs, ys):
    """Spearman by hand: Pearson on ranks. ~10 lines of stdlib, no scipy."""
    def ranks(values):
        order = sorted(range(len(values)), key=lambda i: values[i])
        out = [0.0] * len(values)
        for position, i in enumerate(order):
            out[i] = float(position)
        return out

    rx, ry = ranks(xs), ranks(ys)
    mx, my = statistics.mean(rx), statistics.mean(ry)
    num = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    den = (sum((a - mx) ** 2 for a in rx) ** 0.5
           * sum((b - my) ** 2 for b in ry) ** 0.5)
    return num / den if den else float("nan")


SAMPLE = JUDGEABLE[:12]
scores_a = [geval_score_with(j, JUDGE_A) for j in SAMPLE]
scores_b = [geval_score_with(j, JUDGE_B) for j in SAMPLE]

print("🪞 judge A (%s) mean: %.3f" % (JUDGE_A, statistics.mean(scores_a)))
print("🪞 judge B (%s) mean: %.3f" % (JUDGE_B, statistics.mean(scores_b)))
print("🪞 mean difference (A - B): %+.3f"
      % statistics.mean(a - b for a, b in zip(scores_a, scores_b)))
print("🪞 rank correlation: %.3f" % rank_correlation(scores_a, scores_b))
print()
print("📌 A positive mean difference with a HIGH rank correlation is the benign")
print("   case: judge A is generous, but it orders the replies the same way, so a")
print("   gate comparing two builds still works. A LOW rank correlation is the")
print("   serious one — the two judges do not agree on which build is better.")
if JUDGE_A == JUDGE_B:
    print()
    print("⚠️  JUDGE_B is still set to JUDGE_A, so the numbers above are trivially")
    print("    identical. Point it at a different model family to run the real test.")

🪞 judge A (gpt-4.1-mini) mean: 0.701
🪞 judge B (gpt-4.1-mini) mean: 0.701
🪞 mean difference (A - B): +0.000
🪞 rank correlation: 1.000

📌 A positive mean difference with a HIGH rank correlation is the benign
   case: judge A is generous, but it orders the replies the same way, so a
   gate comparing two builds still works. A LOW rank correlation is the
   serious one — the two judges do not agree on which build is better.

⚠️  JUDGE_B is still set to JUDGE_A, so the numbers above are trivially
    identical. Point it at a different model family to run the real test.


---
## ▶️ See it work

All three judges, both bias probes, one table.

In [13]:
# ============================================================================
# SEE IT WORK: judges by stratum, because judges are biased by stratum too
# ============================================================================
# A judge that scores Hindi replies lower is a very common and very quiet bug:
# it usually means the judge is scoring fluency in ITS strongest language.

JUDGE_COLUMNS = ["geval", "dag", "rubric_without_ref"]


def report_by(rows, dimension):
    buckets = defaultdict(lambda: defaultdict(list))
    for r in rows:
        key = r["strata"].get(dimension)
        if key is None:
            continue
        for c in JUDGE_COLUMNS:
            buckets[key][c].append(r[c])
    return buckets


for dimension in ["lang", "intent"]:
    buckets = report_by(COMPARISON, dimension)
    print("── by %s ──" % dimension)
    print("%-14s %4s %s" % ("", "n", " ".join("%20s" % c for c in JUDGE_COLUMNS)))
    for key in sorted(buckets):
        values = buckets[key]
        cells = " ".join("%20.2f" % statistics.mean(values[c]) for c in JUDGE_COLUMNS)
        print("%-14s %4d %s" % (key, len(values[JUDGE_COLUMNS[0]]), cells))
    print()

print("📌 If one language scores consistently lower across ALL THREE judges, the")
print("   agent is probably worse in that language. If it drops for one judge")
print("   only, you have found a property of the judge, not of the agent.")

── by lang ──
                  n                geval                  dag   rubric_without_ref
en               14                 0.71                 0.71                 1.10
es                6                 0.65                 0.78                 1.39
hi                5                 0.74                 0.94                 1.27

── by intent ──
                  n                geval                  dag   rubric_without_ref
account           6                 0.64                 0.85                 1.22
billing          10                 0.65                 0.90                 1.27
escalation        5                 0.88                 0.88                 1.00
multi_intent      4                 0.69                 0.22                 1.25

📌 If one language scores consistently lower across ALL THREE judges, the
   agent is probably worse in that language. If it drops for one judge
   only, you have found a property of the judge, not of the agent.


---
## 💥 Break it

| Probe | What it should catch |
|---|---|
| `position bias` | Swap the order of two candidate answers and watch a supposedly objective verdict change |
| `same-model bias` | Judge with the same model family that generated the answer, and compare against a different family |

Both probes ran above. The "break" here is not a fault switch — it is that **the judge itself
is the thing under test**, and these are the two cheapest ways to catch it lying.

In [14]:
# ============================================================================
# BREAK IT: make the rubric ambiguous and watch stability collapse
# ============================================================================
# A rubric whose levels overlap gives the judge room to improvise, and a judge
# with room to improvise does not reproduce. This is the most common way a
# working judge quietly stops working: somebody "clarifies" the wording.

VAGUE_RUBRICS = {
    "score1_description": "The reply is not very good.",
    "score2_description": "The reply is fairly good.",
    "score3_description": "The reply is good.",
}

print("Compare the two rubric definitions:")
print()
print("SPECIFIC (yours, above):")
for k in sorted(RUBRICS):
    print("   %s: %s" % (k, RUBRICS[k][:78]))
print()
print("VAGUE (the anti-pattern):")
for k in sorted(VAGUE_RUBRICS):
    print("   %s: %s" % (k, VAGUE_RUBRICS[k]))
print()
print("⚠️  Nothing about the vague version is *wrong*. It is simply not decidable:")
print("    two careful humans reading it would score the same reply differently,")
print("    and so will the same judge on two different days.")
print()
print("📌 The test for a rubric is not 'does it sound reasonable'. It is: can two")
print("    people apply it independently and agree? Notebook 09 measures exactly")
print("    that, with Cohen's kappa, and it is where a vague rubric gets caught.")

Compare the two rubric definitions:

SPECIFIC (yours, above):
   score1_description: TODO: describe a clearly BAD reply — e.g. it answers a different question, or 
   score2_description: TODO: describe a WEAK reply — e.g. broadly correct but vague, or correct and u
   score3_description: TODO: describe a GOOD reply — e.g. answers what was asked, grounded in the doc

VAGUE (the anti-pattern):
   score1_description: The reply is not very good.
   score2_description: The reply is fairly good.
   score3_description: The reply is good.

⚠️  Nothing about the vague version is *wrong*. It is simply not decidable:
    two careful humans reading it would score the same reply differently,
    and so will the same judge on two different days.

📌 The test for a rubric is not 'does it sound reasonable'. It is: can two
    people apply it independently and agree? Notebook 09 measures exactly
    that, with Cohen's kappa, and it is where a vague rubric gets caught.


---
## 🧪 Try it yourself

**Exercise 1 — Write the rubric.**
The `TODO (yours)` in Part 3 is the judge. Fill in the three level descriptions, then answer the
three questions in the comment — especially whether a correct refusal scores at the top of the
scale. Re-run the comparison and see whether your rubric moves closer to the DAG or to GEval.

**Exercise 2 — Run the real same-model test.**
`JUDGE_B` is deliberately set to the same model as `JUDGE_A`, so Part 6 currently proves
nothing. Point it at a different family, re-run in record mode, and report the rank correlation.
Then decide: if the correlation were 0.4, would you still ship a gate built on this judge?

**Exercise 3 — Pick one judge and justify it.**
Notebook 11's gate can only use one quality judge. Choose GEval, the DAG, or the rubric, and
write three sentences defending it to somebody who has to explain a blocked release to an
engineer. "It scored highest" is not a defence.

---
## 📝 Recap and interview angle

### The three shapes, and when each is right

| | `GEval` | `DAGMetric` | RAGAS rubric |
|---|---|---|---|
| You write | prose criteria or steps | an explicit decision tree | fixed level descriptions |
| Reproducibility | lowest | **highest** | high |
| Expresses nuance | **best** | only what you branched for | blunt at boundaries |
| "Why this score?" | a paragraph | **a path** | a level |
| Cost to change | re-word and re-run | add a branch | re-word a level |
| Good for | exploration, fuzzy quality | compliance, auditability | stable regression scoring |

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/judge_comparison.json` | notebook 09 (calibration against human labels) and 11 (the gate picks one) |

### The interview angle
> *"How do you know your LLM judge is any good?"*

The answer is not "we checked a few by hand". It is: the judge is a model, so it gets an
evaluation set of its own, with human labels, and an agreement number. Position bias is the
cheapest and most convincing demonstration — if swapping the order of two answers changes the
verdict, the judge is measuring presentation, not quality, and every score it has ever
produced is suspect.

The follow-up that separates people: *"and what do you do about it?"* Averaging both orders
fixes the symptom. Scoring each answer independently instead of pairwise removes the failure
mode entirely — at the cost of the sensitivity that made pairwise comparison attractive.

In [15]:
# ============================================================================
# SAVE ARTIFACTS: the judge comparison, for calibration and for the gate
# ============================================================================
report = {
    "golden_set_version": GOLDEN["datasheet"]["version"],
    "golden_set_hash": GOLDEN["datasheet"]["content_hash"],
    "n_replies": len(COMPARISON),
    "judges": {
        "geval": round(statistics.mean(r["geval"] for r in COMPARISON), 4),
        "dag": round(statistics.mean(r["dag"] for r in COMPARISON), 4),
        "rubric_without_ref": round(
            statistics.mean(r["rubric_without_ref"] for r in COMPARISON), 4),
    },
    "reference_gap": (
        round(statistics.mean(
            abs(r["rubric_with_ref"] - r["rubric_without_ref"])
            for r in COMPARISON if r["rubric_with_ref"] is not None), 4)
        if any(r["rubric_with_ref"] is not None for r in COMPARISON) else None
    ),
    "high_disagreement_cases": [r["case_id"] for r in COMPARISON if spread(r) > 0.3],
    "bias": {
        "position_flips": flips,
        "position_pairs_tested": len(PAIRS),
        "judge_a": JUDGE_A,
        "judge_b": JUDGE_B,
        "rank_correlation": round(rank_correlation(scores_a, scores_b), 4),
    },
    "by_stratum": {},
    "per_case": [
        {"case_id": r["case_id"], "strata": r["strata"],
         "geval": r["geval"], "dag": r["dag"],
         "rubric_with_ref": r["rubric_with_ref"],
         "rubric_without_ref": r["rubric_without_ref"],
         "spread": round(spread(r), 4)}
        for r in COMPARISON
    ],
}

for dimension in ["lang", "intent"]:
    buckets = report_by(COMPARISON, dimension)
    report["by_stratum"][dimension] = {
        key: {"n": len(values[JUDGE_COLUMNS[0]]),
              **{c: round(statistics.mean(values[c]), 4) for c in JUDGE_COLUMNS}}
        for key, values in buckets.items()
    }

report_path = os.path.join(DATA_DIR, "judge_comparison.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("💾 Wrote", report_path)
print("⚖️  high-disagreement cases queued for human review:",
      len(report["high_disagreement_cases"]))
print("✅ Notebook 08 complete — next: 09_human_feedback_and_calibration.ipynb")

💾 Wrote data/judge_comparison.json
⚖️  high-disagreement cases queued for human review: 21
✅ Notebook 08 complete — next: 09_human_feedback_and_calibration.ipynb
